In [ ]:
# Install required packages.
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Linear, BatchNorm1d
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

os.environ['TORCH'] = torch.__version__
print(torch.__version__)


In [ ]:
!pip install wandb
!pip install captum

import wandb
from captum.attr import IntegratedGradients


from google.colab import userdata
wandb.login(key=userdata.get('WANDB_API_KEY'))

In [ ]:
!pip install torch_geometric
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.datasets import QM9
from torch_geometric.nn import AttentiveFP
from torch_geometric.explain import Explainer, GNNExplainer

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
dataset = QM9(root ="data/qm9")

In [ ]:
print(f"  Number of graphs: {len(dataset)}")
print(f"  Number of node features: {dataset.num_node_features}")
print(f"  Number of edge features: {dataset.num_edge_features}")
print(f"  Number of targets: {dataset.num_classes}")
print(f"  Number of features: {dataset.num_features}")

sample = dataset[0]
print(f"\nExample molecule:")
print(f"  Name       : {sample.name}")
print(f"  Atoms      : {sample.x.shape[0]}")
print(f"  x shape    : {sample.x.shape}   (atoms x features)")
print(f"  edge_index : {sample.edge_index.shape}")
print(f"  edge_attr  : {sample.edge_attr.shape}")
print(f"  pos        : {sample.pos.shape}")
print(f"  y          : {sample.y}")

In [ ]:
from torch.utils.data import random_split

num_graphs = len(dataset)
train_size = int(0.8 * num_graphs)
validation_size = int(0.1 * num_graphs)
test_size = num_graphs - train_size - validation_size

train_dataset, validation_dataset, test_dataset = random_split(
    dataset,
    [train_size, validation_size, test_size]
)


In [ ]:
NUM_TARGETS = 19

TARGET_NAMES = [
    "mu", "alpha", "HOMO", "LUMO", "gap", "R2", "ZPVE",
    "U0", "U", "H", "G", "Cv",
    "U0_atom", "U_atom", "H_atom", "G_atom",
    "A", "B", "C"
]

TARGET_UNITS = [
    "D", "a0^3", "eV", "eV", "eV", "a0^2", "eV",
    "eV", "eV", "eV", "eV", "cal/mol/K",
    "eV", "eV", "eV", "eV",
    "GHz", "GHz", "GHz"
]

In [ ]:
all_train_y = torch.cat([data.y for data in train_dataset], dim=0)

target_mean = all_train_y.mean(dim=0)
target_std  = all_train_y.std(dim=0).clamp(min=1e-6)

print("Per-target statistics (train set):")
print(f"{'idx':>4} {'name':>10} {'mean':>12} {'std':>12} {'unit'}")
print("-" * 55)
for i, (name, unit) in enumerate(zip(TARGET_NAMES, TARGET_UNITS)):
    print(f"{i:>4} {name:>10} {target_mean[i].item():>12.4f} {target_std[i].item():>12.4f}  {unit}")

In [ ]:
def normalize_dataset(subset):
    normalized = []
    for data in subset:
        data.y = (data.y - target_mean) / target_std
        normalized.append(data)
    return normalized

train_dataset      = normalize_dataset(train_dataset)
validation_dataset = normalize_dataset(validation_dataset)
test_dataset       = normalize_dataset(test_dataset)

In [ ]:
batch_size = 64
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=batch_size,
    shuffle=False)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False)

print(f"Train batches : {len(train_loader)}")
print(f"Validation   batches : {len(validation_loader)}")
print(f"Test  batches : {len(test_loader)}")

In [ ]:
model = AttentiveFP(
    in_channels=dataset.num_node_features,
    hidden_channels=64,
    out_channels=NUM_TARGETS,
    edge_dim=dataset.num_edge_features,
    num_layers=2,
    num_timesteps=2
)
model = model.to(device)

print(model)
print(f"\nTrainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.75,
    patience=10,
    min_lr=1e-6
)

_mean = target_mean.to(device)
_std  = target_std.to(device)


def train(loader):
    model.train()
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out  = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
        loss = F.mse_loss(out, batch.y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * batch.num_graphs
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(loader):
    model.eval()
    total_mae = torch.zeros(NUM_TARGETS)

    for batch in loader:
        batch = batch.to(device)
        out   = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)

        pred = out  * _std + _mean
        true = batch.y * _std + _mean

        total_mae += (pred - true).abs().sum(dim=0).cpu()

    mae_per_target = total_mae / len(loader.dataset)
    mean_mae       = mae_per_target.mean().item()
    return mae_per_target, mean_mae

In [ ]:
wandb.init(
    project="qm9-attentivefp",
    config={
        "model": "AttentiveFP",
        "hidden_dim": 64,
        "num_layers": 2,
        "lr": 0.001,
        "batch_size": 64,
        "epochs": 100,
        "optimizer": "Adam"
    }
)

In [ ]:
best_val_mae = float('inf')
best_test_mae_per_target = None

history = {'train_loss': [], 'val_mae': [], 'test_mae': [], 'lr': []}

for epoch in range(1, 100 + 1):
    train_loss              = train(train_loader)
    val_mae_per_t, val_mae  = evaluate(validation_loader)
    test_mae_per_t, test_mae = evaluate(test_loader)

    scheduler.step(val_mae)

    current_lr = optimizer.param_groups[0]['lr']


    history['train_loss'].append(train_loss)
    history['val_mae'].append(val_mae)
    history['test_mae'].append(test_mae)
    history['lr'].append(current_lr)


    log_dict = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_mae": val_mae,
        "test_mae": test_mae,
    }
    for i, name in enumerate(TARGET_NAMES):
        log_dict[f"val_mae_{name}"]  = val_mae_per_t[i].item()
        log_dict[f"test_mae_{name}"] = test_mae_per_t[i].item()

    wandb.log(log_dict)

    if val_mae < best_val_mae:
        best_val_mae             = val_mae
        best_test_mae_per_target = test_mae_per_t.clone()
        torch.save(model.state_dict(), 'best_model.pt')
        wandb.save('best_model.pt')

    if epoch % 10 == 0:
        print(f"Epoch {epoch:>3} | Loss: {train_loss:.4f} | "
              f"Val MAE: {val_mae:.4f} | Test MAE: {test_mae:.4f} | LR: {current_lr:.2e}")

print(f"\nBest mean Val MAE  : {best_val_mae:.4f}")

In [ ]:
print("=" * 45)
print("  PER-TARGET TEST MAE (at best val epoch)")
print("=" * 45)
print(f"{'idx':>4} {'target':>10} {'MAE':>12}")
print("-" * 45)
for i, (name, mae) in enumerate(zip(TARGET_NAMES, best_test_mae_per_target)):
    print(f"{i:>4} {name:>10} {mae.item():>12.4f}")
print("-" * 45)
print(f"{'mean':>15} {best_test_mae_per_target.mean().item():>12.4f}")
print("=" * 45)

In [ ]:
epochs = range(1, 101)

# 1. Training Loss
plt.figure(figsize=(8, 4))
plt.plot(epochs, history['train_loss'], color='steelblue')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss (normalized)')
plt.title('AttentiveFP — QM9 | Training Loss')
plt.grid(True)
plt.tight_layout()
plt.savefig('training_loss.png', dpi=150)
plt.show()

# 2. Val és Test MAE
plt.figure(figsize=(8, 4))
plt.plot(epochs, history['val_mae'],  label='Val MAE', color='steelblue')
plt.plot(epochs, history['test_mae'], label='Test MAE', linestyle='--', color='orange')
plt.xlabel('Epoch')
plt.ylabel('Mean MAE (original units)')
plt.title('AttentiveFP — QM9 | Mean MAE across all 19 targets')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('val_test_mae.png', dpi=150)
plt.show()

# 3. Per-target MAE
plt.figure(figsize=(14, 5))
plt.bar(range(NUM_TARGETS), best_test_mae_per_target.numpy())
plt.xticks(range(NUM_TARGETS),
           [f"{n}\n({u})" for n, u in zip(TARGET_NAMES, TARGET_UNITS)],
           fontsize=8, rotation=45, ha='right')
plt.ylabel('Test MAE')
plt.title('AttentiveFP — QM9 | Per-Target Test MAE')
plt.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig('per_target_mae.png', dpi=150)
plt.show()

wandb.log({"training_loss": wandb.Image("training_loss.png"),
           "val_test_mae": wandb.Image("val_test_mae.png"),
           "per_target_mae":  wandb.Image("per_target_mae.png")})

In [ ]:
plt.plot(history['lr'])
plt.xlabel("Epoch")
plt.ylabel("Learning Rate")
plt.title("Learning Rate Schedule")
plt.grid(True)
plt.show()
plt.savefig("learning_rate_schedule.png", dpi=150)
wandb.log({"learning_rate_schedule": wandb.Image("learning_rate_schedule.png")})

In [ ]:
class SingleTargetWrapper(torch.nn.Module):
    def __init__(self, model, target_idx):
        super().__init__()
        self.model = model
        self.target_idx = target_idx

    def forward(self, x, edge_index, edge_attr, batch):
        out = self.model(x, edge_index, edge_attr, batch)
        return out[:, self.target_idx].unsqueeze(1)

In [ ]:

from collections import defaultdict
import networkx as nx

model.load_state_dict(torch.load('best_model.pt'))
model.eval()

sample = test_dataset[0].to(device)

all_explanations = {}
edge_masks_all = []
node_masks_all = []

for target_idx, target_name in enumerate(TARGET_NAMES):
    wrapper = SingleTargetWrapper(model, target_idx).to(device)

    explainer = Explainer(
        model=wrapper,
        algorithm=GNNExplainer(epochs=200),
        explanation_type='model',
        node_mask_type='attributes',
        edge_mask_type='object',
        model_config=dict(
            mode='regression',
            task_level='graph',
            return_type='raw',
        ),
    )

    exp = explainer(
        x=sample.x,
        edge_index=sample.edge_index,
        edge_attr=sample.edge_attr,
        batch=torch.zeros(sample.x.size(0), dtype=torch.long, device=device),
    )

    all_explanations[target_name] = exp
    edge_masks_all.append(exp.edge_mask.cpu().detach().numpy())
    node_masks_all.append(exp.node_mask.cpu().detach().numpy().squeeze())

edge_masks_all = np.stack(edge_masks_all)
node_masks_all = np.stack(node_masks_all)

In [ ]:
def aggregate_undirected(edge_index, edge_mask):
    edge_groups = defaultdict(list)
    ei = edge_index.cpu().numpy()
    for i in range(ei.shape[1]):
        u, v = int(ei[0, i]), int(ei[1, i])
        edge_groups[tuple(sorted((u, v)))].append(edge_mask[i])
    undir_edges = list(edge_groups.keys())
    undir_weights = np.array([np.mean(v) for v in edge_groups.values()])
    return undir_edges, undir_weights

aggregated_edge_masks = []
for edge_mask in edge_masks_all:
    undir_edges, undir_weights = aggregate_undirected(sample.edge_index, edge_mask)
    aggregated_edge_masks.append(undir_weights)

aggregated_edge_masks = np.stack(aggregated_edge_masks)  # (19, num_undir_edges)

import networkx as nx
G = nx.Graph()
G.add_edges_from(undir_edges)
pos = nx.spring_layout(G, seed=42)

In [ ]:
fig, axes = plt.subplots(4, 5, figsize=(20, 16))
axes = axes.flatten()

for idx, (target_name, unit) in enumerate(zip(TARGET_NAMES, TARGET_UNITS)):
    ax = axes[idx]

    ew = aggregated_edge_masks[idx].copy()
    if ew.max() > 0:
        ew = (ew - ew.min()) / (ew.max() - ew.min() + 1e-8)

    nw = node_masks_all[idx].copy()
    if nw.max() > 0:
        nw = (nw - nw.min()) / (nw.max() - nw.min() + 1e-8)

    nx.draw_networkx_edges(G, pos, ax=ax,
                           edgelist=undir_edges,
                           width=[w * 6 + 0.3 for w in ew],
                           edge_color=ew, edge_cmap=plt.cm.Oranges,
                           edge_vmin=0, edge_vmax=1, alpha=0.9)
    nx.draw_networkx_nodes(G, pos, ax=ax,
                           node_color=nw, cmap=plt.cm.Blues,
                           node_size=250, vmin=0, vmax=1)
    nx.draw_networkx_labels(G, pos, ax=ax, font_size=7)

    ax.set_title(f"{target_name} ({unit})", fontsize=9)
    ax.axis("off")

if len(TARGET_NAMES) < len(axes):
    for j in range(len(TARGET_NAMES), len(axes)):
        axes[j].set_visible(False)

fig.suptitle("GNNExplainer — AttentiveFP on QM9 (per target)", fontsize=14)
plt.tight_layout()
plt.savefig("gnnexplainer_molecules.png", dpi=200)
plt.show()

In [ ]:
wandb.log({
    "explainer/molecules": wandb.Image("gnnexplainer_molecules.png"),
})

In [ ]:
# Edge heatmap
fig1, ax1 = plt.subplots(figsize=(10, 7))
im1 = ax1.imshow(aggregated_edge_masks, aspect="auto", cmap="YlOrRd")
ax1.set_yticks(range(len(TARGET_NAMES)))
ax1.set_yticklabels([f"{n} ({u})" for n, u in zip(TARGET_NAMES, TARGET_UNITS)], fontsize=9)
ax1.set_xlabel("Undirected edge index")
ax1.set_title("Edge importance per target\n(GNNExplainer — AttentiveFP on QM9)")
plt.colorbar(im1, ax=ax1, label="importance")
plt.tight_layout()
plt.savefig("gnnexplainer_edge_heatmap.png", dpi=200)
plt.show()

# Node heatmap
fig2, ax2 = plt.subplots(figsize=(10, 7))
im2 = ax2.imshow(node_masks_all, aspect="auto", cmap="Blues")
ax2.set_yticks(range(len(TARGET_NAMES)))
ax2.set_yticklabels([f"{n} ({u})" for n, u in zip(TARGET_NAMES, TARGET_UNITS)], fontsize=9)
ax2.set_xlabel("Node index")
ax2.set_title("Node importance per target\n(GNNExplainer — AttentiveFP on QM9)")
plt.colorbar(im2, ax=ax2, label="importance")
plt.tight_layout()
plt.savefig("gnnexplainer_node_heatmap.png", dpi=200)
plt.show()

wandb.log({
    "explainer/edge_heatmap": wandb.Image("gnnexplainer_edge_heatmap.png"),
    "explainer/node_heatmap": wandb.Image("gnnexplainer_node_heatmap.png"),
})

In [ ]:
sweep_config = {
    "method": "bayes",
    "metric": {
        "name": "val_loss",
        "goal": "minimize"
    },
    "parameters": {
        "hidden_channels": {"values": [32, 64, 128, 256]},
        "num_layers":      {"values": [2, 3, 4]},
        "num_timesteps":   {"values": [2, 3, 4]},
        "lr":              {"min": 1e-4, "max": 1e-2},
        "batch_size":      {"values": [32, 64, 128]},
    }
}

sweep_id = wandb.sweep(sweep_config, project="QM9_AttentiveFP")
print("Sweep ID:", sweep_id)

In [ ]:
def train_sweep():
    run = wandb.init()
    cfg = run.config

    model = AttentiveFP(
        in_channels=dataset.num_node_features,
        hidden_channels=cfg.hidden_channels,
        out_channels=NUM_TARGETS,
        edge_dim=dataset.num_edge_features,
        num_layers=cfg.num_layers,
        num_timesteps=cfg.num_timesteps,
    ).to(device)

    train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True)
    val_loader   = DataLoader(validation_dataset,   batch_size=cfg.batch_size)

    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5)

    best_val_loss = float('inf')

    for epoch in range(100):
        model.train()
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            out = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
            loss = F.mse_loss(out, batch.y[:, :NUM_TARGETS])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device)
                out = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
                val_loss += F.mse_loss(out, batch.y[:, :NUM_TARGETS]).item()
        val_loss /= len(val_loader)

        scheduler.step(val_loss)
        wandb.log({"val_loss": val_loss, "epoch": epoch})

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), f"best_model_sweep_{run.id}.pt")

In [ ]:
wandb.agent(sweep_id, function=train_sweep, count=30)

In [ ]:
wandb.finish()